### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="horse_colic_survival",
    dataset_year="1989",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C58W23",
    download_description="""
Get the train and test data from UCI.

wget https://archive.ics.uci.edu/static/public/47/horse+colic.zip && unzip horse+colic.zip horse-colic.data horse-colic.test && rm horse+colic.zip && mkdir -p local-data-warehouse/horse_colic_survival && mv horse-colic.data horse-colic.test local-data-warehouse/horse_colic_survival/
""",
    # References
    academic_reference_bibtex="""@misc{McLeish1989HorseColic,
  author       = {McLeish, Mary and Cecile, Matt},
  title        = {{Horse Colic}},
  year         = {1989},
  howpublished = {UCI Machine Learning Repository},
  note         = {{DOI}: https://doi.org/10.24432/C58W23}
}
""",
    academic_reference_bibtex_key="McLeish1989HorseColic",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from UCI and merge the train and test data.

- We encode missing values as NaN.
- We drop features that leak the target.
- We keep only the first instance (hospital number) per horse, as they can be treated multiple times and otherwise leaks information about the target.
- We use the target "outcome" to model a task where we aim to predict if the horse survives or not. We drop cases that have not outcome recorded.
- We keep the duplicates occurring in the data.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="outcome",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="outcome",
)

## Preprocessing

In [ ]:
import pandas as pd

columns = [
    "surgery","age","hospital_number","rectal_temperature","pulse","respiratory_rate","temperature_of_extremities","peripheral_pulse","mucous_membranes","capillary_refill_time","pain","peristalsis","abdominal_distension","nasogastric_tube","nasogastric_reflux","nasogastric_reflux_ph","rectal_exam_feces","abdomen","packed_cell_volume","total_protein","abdominocentesis_appearance","abdominocentesis_total_protein","outcome","surgical_lesion","lesion_type_1","lesion_type_2","lesion_type_3","cp_data"
]

df = pd.read_csv(dataset_mold.path / "horse-colic.data", header=None, names=columns, na_values="?", sep="\s+")
df_test = pd.read_csv(dataset_mold.path / "horse-colic.test", header=None, names=columns, na_values="?", sep="\s+")
df = pd.concat([df, df_test], ignore_index=True)
print("Loaded data shape:", df.shape)

# Drop duplicates by hospital number, keeping the first instance, as they can be treated multiple times and otherwise leaks information about the target.
df = df.drop_duplicates(subset=["hospital_number"], keep="first")

# Only keep non nan outcome rows
df = df[~df["outcome"].isna()]

df = df.drop(columns=[
    "hospital_number", # constant now
    "surgery", # leaks target
    "cp_data", # no significance according to original data description
    # Other target outcomes and thus leaking of outcome
    "surgical_lesion", "lesion_type_1", "lesion_type_2", "lesion_type_3",
])
df["outcome"] = df["outcome"].map({1: "Lived", 2: "Died", 3: "Euthanized"})
as_cat_type = [
    "temperature_of_extremities", "peripheral_pulse", "mucous_membranes", "capillary_refill_time", "pain", "peristalsis", "abdominal_distension", "nasogastric_tube",
    "nasogastric_reflux", "rectal_exam_feces", "abdomen", "abdominocentesis_appearance", "outcome",
]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()